# Puffin Transcription Initiation

![Puffin Transcription Initiation](https://proto-bio.github.io/proto-assets/images/tool/puffin/hero.png)

Puffin predicts per-base transcription initiation signals from DNA sequence and decomposes them into learned motif contributions ([Dudnyk et al., *Science* 2024](https://doi.org/10.1126/science.adj0116)). Three tools share the same model: `puffin-prediction` is a fast forward pass returning all 10 channels, `puffin-interpretation` runs gradient-based motif decomposition for one chosen target signal and strand, and `puffin-gradient` reports a track's mean signal together with its gradient for sequence design.

In [1]:
from proto_tools.utils.notebook_docs import display_api_reference, display_available_tools, display_doc_link, display_docs_section, display_overview

display_doc_link("puffin")
display_overview("puffin")
display_docs_section("puffin", "Background")

# Puffin

Puffin is a sequence-based deep learning model for transcription initiation that predicts per-base initiation signal from DNA and decomposes the prediction into a small set of learned promoter motifs. This toolkit exposes a fast prediction path (`puffin-prediction`), a gradient-based motif-decomposition path (`puffin-interpretation`), and a differentiable track-signal objective (`puffin-gradient`) over the same checkpoint.

In 2024, [Dudnyk et al.](https://doi.org/10.1126/science.adj0116) introduced Puffin, a deep learning model that explains transcription initiation in the human genome at basepair resolution from sequence alone. The model is trained against five [transcription initiation](https://en.wikipedia.org/wiki/Transcription_(biology)) assays (FANTOM CAGE, ENCODE CAGE, ENCODE RAMPAGE, GRO-cap, PRO-cap), each predicted on both strands. The output is a per-base 10-channel signal that can be interpreted as `ln(count_scale_signal + 1)`.

Puffin is structurally constrained: its first convolutional layer plays the role of a learned [motif](https://en.wikipedia.org/wiki/Sequence_motif) filter bank, and the model exposes per-base activation and contribution scores for nine promoter motifs (CREB, ETS, NFY, NRF1, SP, TATA, U1_snRNP, YY1, ZNF143) on each strand. A tenth `Long Inr` filter is used internally by the model to construct the per-base initiator-effect track but is not exposed per-motif. The minimum input is 651 bp because the model uses 325 bp of padding on each side of the predicted output span.

The wrapper accepts raw DNA strings; the upstream coordinate / region / FASTA-file CLI modes (which require an hg38 reference) are intentionally not exposed and callers extract genomic sequences themselves.

## Available tools

In [2]:
display_available_tools("puffin")

- **`run_puffin_gradient()`** — Compute Puffin transcription-initiation gradients for relaxed DNA sequences
- **`run_puffin_interpretation()`** — Motif-level interpretation of transcription initiation with Puffin
- **`run_puffin_prediction()`** — Basepair-resolution transcription initiation prediction with Puffin

### `run_puffin_prediction`

Fast forward pass through Puffin. For each input DNA sequence (>= 651 bp) returns per-base predictions across 10 channels (five transcription assays on both strands). Per-base output length equals `len(sequence) - 650`.

In [3]:
from proto_tools import (
    PuffinPredictionConfig,
    PuffinPredictionInput,
    TRACK_NAMES,
    run_puffin_prediction,
)

In [4]:
# Display input docs
display_api_reference("puffin-prediction", "input", "run_puffin_prediction")

**Input** — `PuffinPredictionInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | DNA sequence(s) >= 651 bp. Per-base output length = len(seq) - 650 |

In [5]:
# Display config docs
display_api_reference("puffin-prediction", "config", "run_puffin_prediction")

**Config** — `PuffinPredictionConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |

In [6]:
from pathlib import Path

# Real HBB promoter window (1650 bp -> 1000 bp per-base output); the HBB
# transcription start site sits at the centre of the scored region.
hbb = "".join(l for l in Path("hbb_genomic.fasta").read_text().splitlines() if not l.startswith(">"))
sequence = hbb[4176:5826]
prediction_result = run_puffin_prediction(
    PuffinPredictionInput(sequences=[sequence]),
    PuffinPredictionConfig(),
)

entry = prediction_result.results[0]
fantom_idx = TRACK_NAMES.index("FANTOM_CAGE+")
peak = max(range(entry.output_length), key=lambda i: entry.predictions[i][fantom_idx])
print(f"shape: {entry.output_length} bp x {len(entry.predictions[0])} channels")
print(f"FANTOM_CAGE+ peak at input position {entry.output_start + peak}")

Running run_puffin_prediction [00:00]

shape: 1000 bp x 10 channels
FANTOM_CAGE+ peak at input position 824


In [7]:
display_api_reference("puffin-prediction", "output", "run_puffin_prediction")

**Output** — `PuffinPredictionOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[PuffinPredictionResult]</code> | required | Per-sequence Puffin prediction results |
| <code>track_names</code> | <code>list[str]</code> | required | Names of the 10 output channels in order |

**`PuffinPredictionResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence</code> | <code>str</code> | required | DNA sequence originally provided to the tool |
| <code>sequence_length</code> | <code>int</code> | required | Length of the provided DNA sequence |
| <code>output_length</code> | <code>int</code> | required | Number of per-base output positions (= sequence_length - 650) |
| <code>output_start</code> | <code>int</code> | required | 0-based sequence coordinate of the first output position |
| <code>output_end</code> | <code>int</code> | required | 0-based exclusive end of the output span in the input sequence |
| <code>predictions</code> | <code>list[list[float]]</code> | required | Per-base log-scale predictions in TRACK_NAMES order (shape [output_length, 10]) |

### `run_puffin_interpretation`

Gradient-based motif decomposition for one chosen target signal and strand. Returns the per-base prediction plus 18 motif activation tracks (9 motifs x 2 strands) and basepair-level contribution scores.

In [8]:
from proto_tools import (
    MOTIF_NAMES,
    PuffinInterpretationConfig,
    PuffinInterpretationInput,
    run_puffin_interpretation,
)

In [9]:
# Display input docs
display_api_reference("puffin-interpretation", "input", "run_puffin_interpretation")

**Input** — `PuffinInterpretationInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | DNA sequence(s) >= 651 bp. Per-base output length = len(seq) - 650 |

In [10]:
# Display config docs
display_api_reference("puffin-interpretation", "config", "run_puffin_interpretation")

**Config** — `PuffinInterpretationConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>target_signal</code> | <code>Literal['FANTOM_CAGE', 'ENCODE_CAGE', 'ENCODE_RAMPAGE', 'GRO_CAP', 'PRO_CAP']</code> | <code>'FANTOM_CAGE'</code> | Which transcription initiation assay's predictions to decompose |
| <code>reverse_strand</code> | <code>bool</code> | <code>False</code> | Decompose the reverse-strand prediction instead of the forward strand |

In [11]:
# Decompose the PRO_CAP forward-strand prediction
sequence = "ATCG" * 412 + "AT"
interpretation_result = run_puffin_interpretation(
    PuffinInterpretationInput(sequences=[sequence]),
    PuffinInterpretationConfig(target_signal="PRO_CAP"),
)

r = interpretation_result.results[0]
print(f"target_signal: {interpretation_result.target_signal}, reverse_strand: {interpretation_result.reverse_strand}")
print(f"motif tracks: {len(r.motif_activations)} (9 motifs x 2 strands)")
print(f"motifs: {MOTIF_NAMES}")

Running run_puffin_interpretation [00:00]

target_signal: PRO_CAP, reverse_strand: False
motif tracks: 18 (9 motifs x 2 strands)
motifs: ('CREB', 'ETS', 'NFY', 'NRF1', 'SP', 'TATA', 'U1_snRNP', 'YY1', 'ZNF143')


In [12]:
display_api_reference("puffin-interpretation", "output", "run_puffin_interpretation")

**Output** — `PuffinInterpretationOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[PuffinInterpretationResult]</code> | required | Per-sequence interpretation results |
| <code>target_signal</code> | <code>str</code> | required | Target transcription initiation signal that was interpreted |
| <code>reverse_strand</code> | <code>bool</code> | required | Whether the reverse-strand head was used |
| <code>motif_names</code> | <code>list[str]</code> | required | The 9 learned motif names (without strand suffix) |

**`PuffinInterpretationResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence</code> | <code>str</code> | required | DNA sequence originally provided to the tool |
| <code>sequence_length</code> | <code>int</code> | required | Length of the provided DNA sequence |
| <code>output_length</code> | <code>int</code> | required | Number of per-base output positions (= sequence_length - 650) |
| <code>output_start</code> | <code>int</code> | required | 0-based sequence coordinate of the first output position |
| <code>output_end</code> | <code>int</code> | required | 0-based exclusive end of the output span in the input sequence |
| <code>prediction</code> | <code>list[float]</code> | required | Predicted log-scale signal for the selected target and strand, length output_length |
| <code>motif_activations</code> | <code>dict[str, list[float]]</code> | required | Per-base motif activation scores keyed by strand-suffixed motif name |
| <code>motif_effects</code> | <code>dict[str, list[float]]</code> | required | Per-base motif effect scores keyed by strand-suffixed motif name |
| <code>sum_motif_effects</code> | <code>list[float]</code> | required | Per-base sum of non-initiator motif effects |
| <code>sum_initiator_effects</code> | <code>list[float]</code> | required | Per-base sum of initiator-motif effects, centered |
| <code>sum_trinucleotide_effects</code> | <code>list[float]</code> | required | Per-base sum of trinucleotide sequence effects, centered |
| <code>sum_total_effects</code> | <code>list[float]</code> | required | Per-base sum of motif, initiator, and trinucleotide effects |
| <code>bp_contribution</code> | <code>list[float]</code> | required | Per-base contribution to transcription initiation |
| <code>bp_contribution_per_motif</code> | <code>dict[str, list[float]]</code> | required | Per-base contribution to transcription, decomposed by motif |
| <code>bp_contribution_to_motif_activation</code> | <code>dict[str, list[float]]</code> | required | Per-base contribution to motif-activation scores, decomposed by motif |

### `run_puffin_gradient`

The gradient tool turns Puffin's initiation prediction into a differentiable quantity. Each term names an output track (with its strand suffix) and a span of output positions, and comes back with that track's mean signal and its own raw gradient; a direction and a weight are goals the caller applies afterwards.

Unlike the fixed-context genomic models, Puffin scores **every base** of a variable-length input — so a candidate is just an `(L, 4)` relaxed matrix and the gradient covers all `L` positions. Inputs need at least 651 bp, since Puffin trims 325 bp from each end and the scored window is `L - 650` positions.

Here the objective targets the FANTOM_CAGE+ peak located above, asking which base changes would raise initiation at exactly that position.

In [13]:
from proto_tools import (
    PuffinGradientConfig, PuffinGradientInput, PuffinSignalTerm, run_puffin_gradient,
)

display_api_reference("puffin-gradient", "input", "run_puffin_gradient")

# Relax the same HBB promoter window: a sharpened one-hot of the real sequence, so the
# starting distribution favours the wild-type base at each position without saturating.
grad_sharpness = 2.0
grad_logits = [
    [grad_sharpness if base == observed else 0.0 for base in "ACGT"]
    for observed in sequence.upper()
]
print(f"relaxed input: {len(grad_logits)} bp -> {len(grad_logits) - 650} scored positions")


**Input** — `PuffinGradientInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>temperature</code> | <code>float &#124; None</code> | <code>1.0</code> | Softmax temperature for softmax(logits / T); None consumes probabilities directly. |
| <code>inputs</code> | <code>list[NucleotideSequenceLogits]</code> | required | One relaxed (L, 4) DNA state per candidate in A,C,G,T order |

relaxed input: 1650 bp -> 1000 scored positions


In [14]:
display_api_reference("puffin-gradient", "config", "run_puffin_gradient")

# Target a 21 bp window centred on the FANTOM_CAGE+ peak found above. Term positions are
# 1-indexed and relative to the scored output window, which is what `peak` already indexes.
grad_config = PuffinGradientConfig(
    gradient_terms=[
        PuffinSignalTerm(
            track="FANTOM_CAGE+",
            position_start=max(1, peak + 1 - 10),
            position_end=min(entry.output_length, peak + 1 + 10),
        )
    ],
)

grad_output = run_puffin_gradient(PuffinGradientInput(inputs=grad_logits), grad_config)


**Config** — `PuffinGradientConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>soft</code> | <code>float</code> | <code>1.0</code> | Blend raw input (0) toward softmax probabilities (1); 1.0 recommended for almost all uses. |
| <code>hard</code> | <code>float</code> | <code>0.0</code> | Straight-through coefficient; 1 forwards a hard one-hot to the model. |
| <code>gradient_terms</code> | <code>list[PuffinSignalTerm]</code> | <code>[PuffinSignalTerm(id='track_signal', track='FANTOM_CAGE+', position_start=None, position_end=None)]</code> | Track-and-position quantities to differentiate, by name or as instances |
| <code>compute_gradient</code> | <code>bool</code> | <code>True</code> | Run backward pass and return gradient; set False for forward-only scoring |

Running run_puffin_gradient [00:00]

In [15]:
display_api_reference("puffin-gradient", "output", "run_puffin_gradient")

grad_result = grad_output.results[0]
grad_term = grad_result.terms[0]
print(f"Scored positions:   {grad_result.metrics['output_length']}")
print(f"Gradient shape:     ({len(grad_term.gradient)}, {len(grad_term.gradient[0])})")
print(f"  {grad_term.term.track} positions "
      f"{grad_term.term.position_start}-{grad_term.term.position_end}: "
      f"mean signal {grad_term.term_value:.4f}")

# The gradient covers the whole input, including the 325 bp flanks Puffin trims from the
# output -- those bases still shape the prediction, so they still carry a derivative.
grad_peak_row = grad_term.gradient[325 + peak]
print(f"Gradient at the peak base: {[round(v, 6) for v in grad_peak_row]}")
display_api_reference("puffin-gradient", "input", "run_puffin_gradient")
display_api_reference("puffin-gradient", "config", "run_puffin_gradient")
display_api_reference("puffin-gradient", "output", "run_puffin_gradient")


**Output** — `PuffinGradientOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[PuffinGradientResult]</code> | required | One result per input candidate, in input order, each carrying its track signals. |
| <code>vocab</code> | <code>list[str]</code> | required | Symbols defining the column ordering for both the input logits and the returned gradient. |

**`PuffinGradientResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>terms</code> | <code>list[PuffinTermGradient]</code> | <code>[]</code> | One gradient per requested track-and-position quantity, in the order requested. |
| <code>metrics</code> | <code>dict[str, Any]</code> | <code>{}</code> | Metrics reported for this item alongside its objective value. |

**`PuffinTermGradient`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>term</code> | <code>PuffinSignalTerm</code> | required | Track-and-position quantity answered. |
| <code>term_value</code> | <code>float</code> | required | Raw scalar the model produced for this quantity. |
| <code>gradient</code> | <code>list[list[float]] &#124; None</code> | <code>None</code> | Raw derivative of term_value w.r.t. the input; matches its shape and column order. |
| <code>grad_l2_norm</code> | <code>float &#124; None</code> | <code>None</code> | Euclidean norm of the flattened gradient; computed automatically when a gradient is present. |

**`PuffinSignalTerm`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>id</code> | <code>Literal['track_signal']</code> | <code>'track_signal'</code> | Identifies this quantity as Puffin mean track signal over a position span. |
| <code>track</code> | <code>Literal['FANTOM_CAGE+', 'ENCODE_CAGE+', 'ENCODE_RAMPAGE+', 'GRO_CAP+', 'PRO_CAP+', 'PRO_CAP-', 'GRO_CAP-', 'ENCODE_RAMPAGE-', 'ENCODE_CAGE-', 'FANTOM_CAGE-']</code> | <code>'FANTOM_CAGE+'</code> | Puffin output track to optimize, named as in TRACK_NAMES (strand suffixed) |
| <code>position_start</code> | <code>int &#124; None</code> | <code>None</code> | First output position to average, 1-indexed inclusive; None starts at the first |
| <code>position_end</code> | <code>int &#124; None</code> | <code>None</code> | Last output position to average, 1-indexed inclusive; None ends at the last |

**`GradientMetadata`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>description</code> | <code>str</code> | <code>''</code> | What this scalar quantity measures. |
| <code>units</code> | <code>str &#124; None</code> | <code>None</code> | Physical units of the value, where it has them. |
| <code>suggested_optimization_direction</code> | <code>Literal['higher', 'lower', 'in-range', 'context-dependent']</code> | <code>'context-dependent'</code> | Advisory direction for this quantity; does not affect the sign of the gradient. |
| <code>vocab</code> | <code>list[str]</code> | required | Column identity and order for this gradient. |
| <code>coordinate_space</code> | <code>Literal['logits', 'probabilities']</code> | required | Whether the derivative is w.r.t. logits or probabilities. |
| <code>length</code> | <code>int</code> | required | Number of differentiated rows. |
| <code>positions</code> | <code>list[int] &#124; None</code> | <code>None</code> | 1-indexed input position per row; None means contiguous 1..length. |
| <code>input_fingerprint</code> | <code>str</code> | required | Digest of the differentiated matrix, identifying the point the gradient was taken at. |
| <code>hard</code> | <code>float</code> | <code>0.0</code> | Straight-through coefficient in force. |
| <code>soft</code> | <code>float</code> | <code>1.0</code> | Relaxation blend in force. |

**Metrics** — `PuffinGradientSampleMetrics` (one set per `results` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>output_length</code> **(primary)** | <code>int</code> | <code>&gt;= 1</code> |  | Number of scored output positions for this candidate. |

Scored positions:   1000
Gradient shape:     (1650, 4)
  FANTOM_CAGE+ positions 490-510: mean signal 0.5629
Gradient at the peak base: [-0.015269, 0.003168, 0.005469, 0.006633]


**Input** — `PuffinGradientInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>temperature</code> | <code>float &#124; None</code> | <code>1.0</code> | Softmax temperature for softmax(logits / T); None consumes probabilities directly. |
| <code>inputs</code> | <code>list[NucleotideSequenceLogits]</code> | required | One relaxed (L, 4) DNA state per candidate in A,C,G,T order |

**Config** — `PuffinGradientConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>soft</code> | <code>float</code> | <code>1.0</code> | Blend raw input (0) toward softmax probabilities (1); 1.0 recommended for almost all uses. |
| <code>hard</code> | <code>float</code> | <code>0.0</code> | Straight-through coefficient; 1 forwards a hard one-hot to the model. |
| <code>gradient_terms</code> | <code>list[PuffinSignalTerm]</code> | <code>[PuffinSignalTerm(id='track_signal', track='FANTOM_CAGE+', position_start=None, position_end=None)]</code> | Track-and-position quantities to differentiate, by name or as instances |
| <code>compute_gradient</code> | <code>bool</code> | <code>True</code> | Run backward pass and return gradient; set False for forward-only scoring |

**Output** — `PuffinGradientOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[PuffinGradientResult]</code> | required | One result per input candidate, in input order, each carrying its track signals. |
| <code>vocab</code> | <code>list[str]</code> | required | Symbols defining the column ordering for both the input logits and the returned gradient. |

**`PuffinGradientResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>terms</code> | <code>list[PuffinTermGradient]</code> | <code>[]</code> | One gradient per requested track-and-position quantity, in the order requested. |
| <code>metrics</code> | <code>dict[str, Any]</code> | <code>{}</code> | Metrics reported for this item alongside its objective value. |

**`PuffinTermGradient`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>term</code> | <code>PuffinSignalTerm</code> | required | Track-and-position quantity answered. |
| <code>term_value</code> | <code>float</code> | required | Raw scalar the model produced for this quantity. |
| <code>gradient</code> | <code>list[list[float]] &#124; None</code> | <code>None</code> | Raw derivative of term_value w.r.t. the input; matches its shape and column order. |
| <code>grad_l2_norm</code> | <code>float &#124; None</code> | <code>None</code> | Euclidean norm of the flattened gradient; computed automatically when a gradient is present. |

**`PuffinSignalTerm`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>id</code> | <code>Literal['track_signal']</code> | <code>'track_signal'</code> | Identifies this quantity as Puffin mean track signal over a position span. |
| <code>track</code> | <code>Literal['FANTOM_CAGE+', 'ENCODE_CAGE+', 'ENCODE_RAMPAGE+', 'GRO_CAP+', 'PRO_CAP+', 'PRO_CAP-', 'GRO_CAP-', 'ENCODE_RAMPAGE-', 'ENCODE_CAGE-', 'FANTOM_CAGE-']</code> | <code>'FANTOM_CAGE+'</code> | Puffin output track to optimize, named as in TRACK_NAMES (strand suffixed) |
| <code>position_start</code> | <code>int &#124; None</code> | <code>None</code> | First output position to average, 1-indexed inclusive; None starts at the first |
| <code>position_end</code> | <code>int &#124; None</code> | <code>None</code> | Last output position to average, 1-indexed inclusive; None ends at the last |

**`GradientMetadata`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>description</code> | <code>str</code> | <code>''</code> | What this scalar quantity measures. |
| <code>units</code> | <code>str &#124; None</code> | <code>None</code> | Physical units of the value, where it has them. |
| <code>suggested_optimization_direction</code> | <code>Literal['higher', 'lower', 'in-range', 'context-dependent']</code> | <code>'context-dependent'</code> | Advisory direction for this quantity; does not affect the sign of the gradient. |
| <code>vocab</code> | <code>list[str]</code> | required | Column identity and order for this gradient. |
| <code>coordinate_space</code> | <code>Literal['logits', 'probabilities']</code> | required | Whether the derivative is w.r.t. logits or probabilities. |
| <code>length</code> | <code>int</code> | required | Number of differentiated rows. |
| <code>positions</code> | <code>list[int] &#124; None</code> | <code>None</code> | 1-indexed input position per row; None means contiguous 1..length. |
| <code>input_fingerprint</code> | <code>str</code> | required | Digest of the differentiated matrix, identifying the point the gradient was taken at. |
| <code>hard</code> | <code>float</code> | <code>0.0</code> | Straight-through coefficient in force. |
| <code>soft</code> | <code>float</code> | <code>1.0</code> | Relaxation blend in force. |

**Metrics** — `PuffinGradientSampleMetrics` (one set per `results` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>output_length</code> **(primary)** | <code>int</code> | <code>&gt;= 1</code> |  | Number of scored output positions for this candidate. |

## Export Results

In [16]:
from pathlib import Path

out_dir = Path("puffin_output")
out_dir.mkdir(exist_ok=True)
prediction_result.export(name="prediction", export_path=out_dir, file_format="json")
interpretation_result.export(name="interpretation", export_path=out_dir, file_format="json")